# Train model

In [ ]:
import sys
from pathlib import Path

# Kaggle:
# sys.path.insert(0, "/kaggle/input/datasets/gpla77/pro5-code")
# DATA_PATH = Path("/kaggle/input/datasets/gpla77/pro5-data/train.npz")
# CKPT_DIR  = Path("/kaggle/working/checkpoints")

# Local:
DATA_PATH = Path("data/train.npz")
CKPT_DIR  = Path("checkpoints")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Data   : {DATA_PATH}  (exists: {DATA_PATH.exists()})")

# Dataset

In [ ]:
from dataset import MotionDataset

dataset = MotionDataset(str(DATA_PATH))
seq, label = dataset[0]
print(f"Samples        : {len(dataset)}")
print(f"Sequence shape : {seq.shape}")   # [T, J, 3]
print(f"Num classes    : {int(dataset.labels.max()) + 1}")

# Train

In [ ]:
from train import train

model = train(
    dataset        = dataset,
    # model
    d_model        = 256,
    nhead          = 4,
    num_layers     = 4,
    dropout        = 0.1,
    # diffusion
    timesteps      = 1000,
    beta_start     = 1e-4,
    beta_end       = 0.02,
    cfg_drop_prob  = 0.1,
    guidance_scale = 3.0,
    # training
    epochs         = 3,
    batch_size     = 32,
    lr             = 1e-4,
    optimizer      = "adamw",
    weight_decay   = 1e-4,
    scheduler      = "cosine",
    grad_clip      = 1.0,
    save_every     = 1,
    # qualitative eval
    eval_every     = 1,
    eval_samples   = 4,
    # misc
    device         = DEVICE,
    ckpt_dir       = str(CKPT_DIR),
    resume_from    = None,
    # Kaggle resume:
    # resume_from  = "/kaggle/input/YOUR_MODEL_DATASET/ckpt_e050.pt",
    seed           = 42,
    num_workers    = 0,   
)

In [ ]:
from pathlib import Path
from sample import visualize_training_samples

for pt_file in sorted(Path("checkpoints/samples").glob("samples_e*.pt")):
    visualize_training_samples(
        samples_pt = str(pt_file),
        save_dir   = f"checkpoints/samples/gifs/{pt_file.stem}",
        fps        = 24,
    )